# Basin allocation and reconstruction

Inputs (`{b}` = `closed`, `open`):
- `output/central_pathways.csv`: design pathways (notebook 01)
- `input/co2block_basin_resources_{b}.csv`: CO2BLOCK basin injection rate, well number and storage resource for injection periods of 10 to 150 years
- `input/co2block_well_configurations_{b}.csv.gz`: CO2BLOCK well arrays used to refine the last allocated basin
- `input/basin_properties_{b}.xlsx`: CO2BLOCK basin properties (basin order)

## Setup

In [ ]:
import io
import math
import time
from pathlib import Path

import numpy as np
import pandas as pd
from scipy.optimize import brentq
from scipy.special import expit

ROOT = Path.cwd().resolve()
if not (ROOT / 'input').is_dir() and (ROOT.parent / 'input').is_dir():
    ROOT = ROOT.parent
assert (ROOT / 'input').is_dir(), 'Run from the Global Study folder or its code folder.'
INPUT, OUTPUT = ROOT / 'input', ROOT / 'output'
BOUNDARIES = ['closed', 'open']
ORDERS = ['descend', 'ascend']
KEYS = ['scenario', 'country', 'model']
CASE = KEYS + ['allocation_order']
KEY = ['boundary'] + CASE

T0, T_END = 2030, 2180
YEARS = np.arange(T0, T_END + 1)
DURATION_YR = T_END - T0
PERIODS = list(range(10, DURATION_YR + 1, 10))

## Design pathways

In [ ]:
central = pd.read_csv(OUTPUT / 'central_pathways.csv', float_precision='round_trip')
assert len(central) == 120 and not central.duplicated(KEYS).any()

def anchored_curve_values(model, C, r, S0, year):
    tau = np.asarray(year) - T0
    if model == 'Logistic':
        z = np.log((C - S0) / S0) - r * tau
        return C * expit(-z), C * r * expit(-z) * expit(z)
    u = np.log(C / S0) * np.exp(-r * tau)
    return C * np.exp(-u), C * r * u * np.exp(-u)

def demand_curve(rate_mt_yr):
    # The allocation reads demand from CSV text with the default parser; keep that last-bit rounding.
    text = pd.DataFrame({'year': YEARS, 'total_rate': rate_mt_yr}).to_csv(index=False)
    curve = pd.read_csv(io.StringIO(text))
    return curve['year'].to_numpy(float), curve['total_rate'].to_numpy(float)

curves = {}
for row in central.to_dict('records'):
    rate = anchored_curve_values(row['model'], row['C_scenario_Gt'], row['r_per_yr'], row['S0_Gt'], YEARS)[1]
    assert np.isfinite(rate).all() and (rate >= 0).all()
    curves[tuple(row[k] for k in KEYS)] = demand_curve(rate * 1000.0)

## Basin inputs

In [ ]:
def load_basins(boundary):
    resources = pd.read_csv(INPUT / f'co2block_basin_resources_{boundary}.csv')
    workbook = pd.read_excel(INPUT / f'basin_properties_{boundary}.xlsx')
    meta = resources[['Region_no', 'Region_name', 'AssignedCountry', 'RegionGroup', 'InStudyScope']]
    meta = meta.drop_duplicates().sort_values('Region_no')
    assert resources['BoundaryCondition'].eq(boundary).all()
    assert workbook['Domain type'].str.strip().str.lower().eq(boundary).all()
    assert meta['Region_name'].tolist() == workbook['Basin Name'].tolist()
    study = resources[resources['InStudyScope']]
    matrix = lambda value: (study.pivot(index=['Region_no', 'Region_name'], columns='Period_yr', values=value)
                            .reindex(columns=PERIODS).sort_index())
    configs = pd.read_csv(INPUT / f'co2block_well_configurations_{boundary}.csv.gz', float_precision='round_trip')
    configs = {key: (g['Number_of_Wells'].to_numpy(), g['Q_per_well_Mt_yr'].to_numpy())
               for key, g in configs.groupby(['Region_no', 'Period_yr'])}
    return dict(boundary=boundary, meta=meta[meta['InStudyScope']], configs=configs,
                rates=matrix('Region_Q_Mt_yr'), sites=matrix('Number_of_Wells'))

def jurisdiction_basins(meta, country):
    if country == 'US':
        mask, basis = meta['AssignedCountry'].eq('USA'), "AssignedCountry in ['USA']"
    elif country == 'EU':
        mask = meta['RegionGroup'].eq('Europe') & ~meta['AssignedCountry'].eq('UK')
        basis = "RegionGroup in ['Europe'] and AssignedCountry not in ['UK']"
    elif country == 'Middle East':
        mask, basis = meta['RegionGroup'].eq('Middle East'), "RegionGroup in ['Middle East']"
    else:
        mask, basis = meta['AssignedCountry'].eq(country), f'AssignedCountry == {country}'
    ids = meta.loc[mask, 'Region_no'].astype(int).tolist()
    assert ids, country
    return ids, basis

## Allocation

In [ ]:
def refine_last_basin(configs, region_no, period, remaining_rate, remaining_volume):
    if (region_no, period) not in configs:
        raise KeyError(f'No CO2BLOCK well configurations for basin {region_no}, {period}-yr period.')
    wells, q = configs[(region_no, period)]
    rate = q * wells
    ok = (rate > remaining_rate) & (rate * period / 1000.0 * 1000.0 > remaining_volume)
    if not ok.any():
        return None
    k = int(np.argmax(ok))
    return float(rate[k]), float(wells[k])

def allocate(rates, sites, years, demand, order, configs):
    rate_m, site_m = rates.to_numpy(float), sites.to_numpy(float)
    region_no = rates.index.get_level_values('Region_no').to_numpy(int)
    region_name = rates.index.get_level_values('Region_name').to_numpy(object)
    totals = rate_m.sum(axis=1)
    sort_idx = np.argsort(-totals if order == 'descend' else totals, kind='stable')
    peak, total_volume = float(np.max(demand)), float(np.trapezoid(demand, years))
    remain, t_remaining = demand.copy(), DURATION_YR
    alloc_total, cum_vol, starts, ends, rows = [], [], [], [], []
    for step, j in enumerate(sort_idx):
        longer = np.where(np.array(PERIODS, float) >= t_remaining)[0]
        idx = int(longer[0]) if len(longer) else len(PERIODS) - 1
        vol_possible = np.array(PERIODS[:idx + 1], float) * rate_m[j, :idx + 1]
        prev_total = alloc_total[-1] if alloc_total else 0.0
        prev_vol = cum_vol[-1] if cum_vol else 0.0
        step_rate, step_total, period, n_sites = 0.0, prev_total, float(PERIODS[idx]), 0.0
        curve_start = curve_end = math.nan
        accepted = breaking = last_basin = False
        status, refined = 'not_required', None
        for i in range(idx, -1, -1):
            test = remain - rate_m[j, i]
            nonneg = np.where(test >= 0)[0]
            provisional = prev_total + rate_m[j, i]
            if len(nonneg):
                left, right = nonneg[0], nonneg[-1]
                start = (years[left - 1] + (provisional - demand[left - 1]) * (years[left] - years[left - 1])
                         / (demand[left] - demand[left - 1]) if years[left] > years[0] else float(years[0]))
                end = (years[right] + (provisional - demand[right]) * (years[right + 1] - years[right])
                       / (demand[right + 1] - demand[right]) if years[right] < years[-1] else years[-1])
                area_t = np.concatenate(([start], years[left:right + 1], [end]))
                area_q = np.concatenate(([0.0], test[nonneg], [0.0]))
                area = total_volume - prev_vol - float(np.trapezoid(area_q, area_t))
            else:
                area = total_volume - prev_vol
            if (vol_possible[i] > area or abs(vol_possible[i] - area) < 0.01) and rate_m[j, i] >= step_rate:
                accepted = True
                step_rate, step_total = float(rate_m[j, i]), float(provisional)
                period, n_sites = float(PERIODS[i]), float(site_m[j, i])
                if len(nonneg):
                    curve_start, curve_end = float(start), float(end)
                cum_volume = float(prev_vol + area)
                if provisional >= peak:
                    last_basin = True
                    status, refined = 'no_valid_scenario', False
                    choice = refine_last_basin(configs, int(region_no[j]), PERIODS[i], float(peak - prev_total), area)
                    if choice:
                        status, refined = 'refined', True
                        step_rate, n_sites = choice
                        step_total = float(prev_total + step_rate)
                if not len(nonneg):
                    breaking = True
            if last_basin:
                break
        remain = remain - step_rate
        alloc_total.append(step_total)
        cum_vol.append(cum_volume if accepted else 0.0)
        rows.append({
            'Step': step + 1, 'Region no': int(region_no[j]), 'Region name': str(region_name[j]),
            'Start [y]': float(years[0]) if step == 0 else starts[-1],
            'End [y]': float(years[0]) + DURATION_YR if step == 0 else ends[-1],
            'Duration [y]': period, 'Step rate increment [Mt/y]': step_rate,
            'Step rate cumulative [Mt/y]': step_total, 'No_sites': n_sites,
            'Curve start [y]': curve_start, 'Curve end [y]': curve_end,
            'allocation_time_basis': 'elapsed_years_151_nodes_150_intervals',
            'final_region_refinement_attempted': status != 'not_required',
            'final_region_refined': refined, 'final_region_refinement_status': status})
        if breaking:
            break
        starts.append(curve_start)
        ends.append(curve_end)
        t_remaining = curve_end - curve_start
    return pd.DataFrame(rows)

## Screening

In [ ]:
MARKERS = {
    'diagnostic_basis': 'allocation_level_piecewise_linear_v1',
    'diagnostic_start_year': T0, 'diagnostic_end_year': T_END,
    'diagnostic_elapsed_years': float(DURATION_YR), 'diagnostic_annual_nodes': len(YEARS),
    'shortfall_count_basis': 'annual_nodes',
    'diagnostic_integration': 'piecewise_linear_with_level_crossings',
    'initial_stock_included': False,
}

def diagnose(assignment, years, demand, checkpoints=(2050, 2100, 2180)):
    level = float(assignment['Step rate cumulative [Mt/y]'].iloc[-1])
    peak = float(np.max(demand))
    passed = level >= peak
    satisfied, gap = np.minimum(demand, level), np.maximum(demand - level, 0.0)
    masses, short_duration = np.zeros((len(years), 3)), 0.0
    for i in range(1, len(years)):
        t0, t1, d0, d1 = years[i - 1], years[i], demand[i - 1], demand[i]
        ts, ds = [t0, t1], [d0, d1]
        if (d0 < level < d1) or (d1 < level < d0):
            crossing = t0 + (t1 - t0) * (level - d0) / (d1 - d0)
            ts, ds = [t0, crossing, t1], [d0, level, d1]
        q, dt = np.asarray(ds), np.diff(ts)
        short_duration += float(np.sum(dt[(q[:-1] + q[1:]) / 2.0 > level]))
        masses[i] = [np.sum(dt * (v[:-1] + v[1:]) / 2.0) / 1000.0
                     for v in (q, np.minimum(q, level), np.maximum(q - level, 0.0))]
    cum = np.cumsum(masses, axis=0)
    annual = pd.DataFrame({
        'year': years.astype(int), 'demand_rate_mt_yr': demand, 'raw_rate_mt_yr': demand,
        'allocation_level_mt_yr': level, 'satisfied_rate_mt_yr': satisfied, 'screened_rate_mt_yr': satisfied,
        'gap_rate_mt_yr': gap, 'raw_cumulative_gt': cum[:, 0], 'satisfied_cumulative_gt': cum[:, 1],
        'screened_cumulative_gt': cum[:, 1], 'unmet_cumulative_gt': cum[:, 2], 'gap_cumulative_gt': cum[:, 2],
        **MARKERS})
    short = years[demand > level].astype(int)
    k = int(np.argmax(gap))
    summary = {
        'screening_criterion': 'matlab_final_allocated_rate_ge_peak_demand',
        'final_allocated_rate_mt_yr': level, 'allocated_peak_mt_yr': level, 'peak_demand_mt_yr': peak,
        'rate_margin_mt_yr': level - peak, 'allocation_pass': passed, 'whole_period_pass': passed, **MARKERS,
        'achieved_peak_mt_yr': float(np.max(satisfied)), 'total_demand_volume_gt': float(cum[-1, 0]),
        'satisfied_demand_volume_gt': float(cum[-1, 1]), 'total_unmet_volume_gt': float(cum[-1, 2]),
        'years_of_shortfall': len(short), 'shortfall_node_count': len(short),
        'shortfall_duration_yr': short_duration,
        'first_shortfall_year': int(short[0]) if len(short) else None,
        'last_shortfall_year': int(short[-1]) if len(short) else None,
        'shortfall_window_yr': int(short[-1] - short[0] + 1) if len(short) else 0,
        'max_gap_rate_mt_yr': float(gap[k]), 'max_gap_mt_yr': float(gap[k]),
        'max_gap_year': int(years[k]) if len(short) else None}
    for year in checkpoints:
        raw, met, unmet = cum[int(np.flatnonzero(years == year)[0])]
        summary.update({f'raw_cum_gt_{year}': float(raw), f'screened_cum_gt_{year}': float(met),
                        f'cum_gap_gt_{year}': float(unmet),
                        f'cum_gap_pct_{year}': float(100 * unmet / raw) if raw > 0 else 0.0})
    return summary, annual

def screen(basins, row, order, rate_mt_yr=None, extra=None):
    ids, basis = jurisdiction_basins(basins['meta'], row['country'])
    years, demand = curves[tuple(row[k] for k in KEYS)] if rate_mt_yr is None else demand_curve(rate_mt_yr)
    pick = basins['rates'].index.get_level_values('Region_no').isin(ids)
    assignment = allocate(basins['rates'][pick], basins['sites'][pick], years, demand, order, basins['configs'])
    diagnostics, annual = diagnose(assignment, years, demand)
    n = assignment['final_region_refinement_status'].value_counts()
    refined, no_valid = int(n.get('refined', 0)), int(n.get('no_valid_scenario', 0))
    case = {**{k: row[k] for k in KEYS}, 'boundary': basins['boundary'], 'allocation_order': order}
    summary = {**case, **(extra or {}), 'basin_count': len(ids), 'basin_match_basis': basis, **diagnostics,
               'final_region_refinement_attempted_count': refined + no_valid,
               'final_region_refined_count': refined,
               'final_region_refinement_no_valid_scenario_count': no_valid,
               'final_region_refinement_not_required_count': int(n.get('not_required', 0))}
    return summary, assignment.assign(**case), annual.assign(**case)

def run_screening(basins):
    out = OUTPUT / basins['boundary']
    out.mkdir(parents=True, exist_ok=True)
    summaries, allocations, paths = zip(*[screen(basins, row, order) for order in ORDERS
                                          for row in central.to_dict('records')])
    pd.DataFrame(summaries).to_csv(out / 'screening_summary.csv', index=False)
    pd.concat(allocations, ignore_index=True).to_csv(out / 'basin_allocation.csv', index=False)
    pd.concat(paths, ignore_index=True).to_csv(out / 'screened_paths.csv', index=False)

## Reconstruction equations

In [ ]:
PHI = (3.0 + math.sqrt(5.0)) / 2.0
SQRT3 = math.sqrt(3.0)
NAN_FIELDS = ('C_recon_order_gt', 'r_recon_order', 'g_recon_order', 'rate_2050_recon_order',
              'peak_rate_recon_order', 'peak_year_recon_order', 'tp_recon_order', 'tn_recon_order')

def logistic_rate_at(y, C, r, S0, t0=T0):
    k = (C - S0) / S0
    S = C / (1.0 + k * np.exp(-r * (y - t0)))
    return r * S * (1.0 - S / C)

def gompertz_rate_at(y, C, b, r, t0=T0):
    s = b * np.exp(-r * (y - t0))
    return C * r * s * np.exp(-s)

def logistic_peak_year(C, r, S0, t0=T0):
    return t0 + math.log((C - S0) / S0) / r

def gompertz_peak_year(C, S0, r, t0=T0):
    return t0 + math.log(-math.log(S0 / C)) / r

def analytic_stock(model, C, S0, r, years=YEARS):
    if model == 'Logistic':
        return C / (1.0 + ((C - S0) / S0) * np.exp(-r * (years - T0)))
    return C * np.exp(math.log(S0 / C) * np.exp(-r * (years - T0)))

def forward_r(model, C, S0, r, years=YEARS):
    if model == 'Logistic':
        rate, tp = logistic_rate_at(years, C, r, S0), logistic_peak_year(C, r, S0)
    else:
        rate, tp = gompertz_rate_at(years, C, -math.log(S0 / C), r), gompertz_peak_year(C, S0, r)
    return {'rate': rate, 'cum': analytic_stock(model, C, S0, r, years), 'tp': tp}

def r_to_g(model, C, S0, r):
    if model == 'Logistic':
        k = (C - S0) / S0
        S_Tn = C / (3.0 + SQRT3)
        A = math.log(k) - math.log(2.0 + SQRT3)
        if A <= 0 or math.log(S_Tn / S0) <= 0:
            return float('nan')
        return math.exp(r * math.log(S_Tn / S0) / A) - 1.0
    b = -math.log(S0 / C)
    P_Tn = C * math.exp(-PHI)
    B = math.log(b / PHI)
    if B <= 0 or math.log(P_Tn / S0) <= 0:
        return float('nan')
    return math.exp(r * math.log(P_Tn / S0) / B) - 1.0

def C_of_r(model, R_peak, r):
    return (4.0 * R_peak / r) if model == 'Logistic' else (math.e * R_peak / r)

def peak_year_of(model, C, S0, r):
    return logistic_peak_year(C, r, S0) if model == 'Logistic' else gompertz_peak_year(C, S0, r)

def rate_at_2050(model, C, S0, r):
    if model == 'Logistic':
        return float(logistic_rate_at(2050.0, C, r, S0))
    return float(gompertz_rate_at(2050.0, C, -math.log(S0 / C), r))

def r_upper_bound(model, S0, R_peak):
    ratio = (3.0 + SQRT3) if model == 'Logistic' else math.exp(PHI)
    coef = 4.0 if model == 'Logistic' else math.e
    return coef * R_peak / (S0 * ratio * 1.01)

def solve_reconstruction(row, env):
    model, S0 = row['model'], row['S_0']
    R2050, R_peak = float(row['rate_2050_original']), float(row['R_peak_order'])
    unsolved = lambda reason: {'landmark_infeasible': True, 'landmark_reason': reason, **{c: np.nan for c in NAN_FIELDS}}
    if R_peak < R2050 * (1.0 - 1e-6):
        return unsolved('R_peak < R2050')
    r_grid = np.geomspace(1e-5, r_upper_bound(model, S0, R_peak) * 0.999, 600)
    f = []
    for r in r_grid:
        C = C_of_r(model, R_peak, r)
        if C <= S0:
            f.append(np.nan)
            continue
        try:
            f.append(rate_at_2050(model, C, S0, r) - R2050)
        except (ValueError, OverflowError):
            f.append(np.nan)
    f = np.asarray(f)
    valid = ~np.isnan(f)
    if not valid.any():
        return unsolved('no valid r grid')
    roots = []
    for i in range(len(r_grid) - 1):
        if valid[i] and valid[i + 1] and f[i] * f[i + 1] < 0:
            try:
                roots.append(brentq(lambda r: rate_at_2050(model, C_of_r(model, R_peak, r), S0, r) - R2050,
                                    r_grid[i], r_grid[i + 1], rtol=1e-7))
            except (ValueError, RuntimeError):
                pass
    if not roots:
        return unsolved('no root in scan')
    post = [(rr, C_of_r(model, R_peak, rr), peak_year_of(model, C_of_r(model, R_peak, rr), S0, rr)) for rr in roots]
    post = [(rr, C, tp) for rr, C, tp in post if tp >= 2050.0]
    if not post:
        return unsolved('all roots have peak < 2050')
    if len(post) > 1:
        post.sort(key=lambda t: float(np.sqrt(np.mean((forward_r(model, t[1], S0, t[0])['rate'] - env) ** 2))))
    r_sel, C_sel, tp_sel = post[0]
    fwd = forward_r(model, C_sel, S0, r_sel)
    if model == 'Logistic':
        tn_sel = T0 + (math.log((C_sel - S0) / S0) - math.log(2.0 + SQRT3)) / r_sel
    else:
        tn_sel = T0 + math.log(-math.log(S0 / C_sel) / PHI) / r_sel
    return {'landmark_infeasible': False, 'landmark_reason': '',
            'C_recon_order_gt': C_sel, 'r_recon_order': r_sel, 'g_recon_order': r_to_g(model, C_sel, S0, r_sel),
            'rate_2050_recon_order': rate_at_2050(model, C_sel, S0, r_sel), 'peak_rate_recon_order': R_peak,
            'peak_year_recon_order': int(YEARS[int(np.argmax(fwd['rate']))]),
            'tp_recon_order': tp_sel, 'tn_recon_order': tn_sel}

## Reconstruction and rescreening

In [ ]:
EXCEED_EPS_GT = 1e-4

def reconstruct(basins):
    out = OUTPUT / basins['boundary']
    read = lambda name: pd.read_csv(out / f'{name}.csv', float_precision='round_trip')
    summary, annual = read('screening_summary'), read('screened_paths')
    envelopes = {case: path.sort_values('year') for case, path in annual.groupby(CASE, sort=False)}
    params = central.set_index(KEYS)
    rows, paths = [], []
    for original in summary.loc[~summary['allocation_pass']].to_dict('records'):
        case = tuple(original[k] for k in CASE)
        p, path = params.loc[case[:3]], envelopes[case]
        env = path['screened_rate_mt_yr'].to_numpy(float) / 1000.0
        raw = path['raw_rate_mt_yr'].to_numpy(float) / 1000.0
        level = float(original['final_allocated_rate_mt_yr'])
        assert float(original['achieved_peak_mt_yr']) == level
        row = {k: original[k] for k in KEY}
        row.update(pathway='reconstructed', S0_gt=float(p.S0_Gt), C_scenario_gt=float(p.C_scenario_Gt),
                   g_cap=float(p.g_cap), Type='B_fixed' if p.target_type == 'fixed' else 'A',
                   g_original=float(p.g_star), r_original=float(p.r_per_yr), tn_original=float(p.tn),
                   tp_original=float(p.tp), rate_2050_original=float(raw[YEARS == 2050][0]),
                   R_peak_order=level / 1000.0, Y_peak_order=int(YEARS[np.argmax(env)]),
                   original_peak_rate_Gt_yr=float(raw.max()), original_peak_year=int(YEARS[np.argmax(raw)]),
                   C_recon_includes_initial_stock=True)
        row.update({f'original_{field}': value for field, value in original.items() if field not in KEY})
        row.update(solve_reconstruction({**row, 'S_0': row['S0_gt']}, env))
        row['reconstruction_failure_class'] = (
            'solved' if not row['landmark_infeasible'] else
            'mathematical_target_above_peak' if row['landmark_reason'] == 'R_peak < R2050' else
            'numerical_or_domain_unresolved')
        forward = None
        if not row['landmark_infeasible']:
            forward = forward_r(row['model'], row['C_recon_order_gt'], row['S0_gt'], row['r_recon_order'])
            difference = (forward['rate'] - env) * 1000.0
            exceeded = (forward['rate'] - env) > EXCEED_EPS_GT
            count = int(exceeded.sum())
            row.update(max_envelope_exceedance_mt_yr=float(np.max(difference)), years_exceeding_envelope=count,
                       first_exceed_year=int(YEARS[exceeded.argmax()]) if count else np.nan,
                       last_exceed_year=int(YEARS[len(YEARS) - 1 - exceeded[::-1].argmax()]) if count else np.nan,
                       rmse_to_envelope_mt_yr=float(np.sqrt(np.mean(difference ** 2))),
                       envelope_consistent=bool(count == 0))
        else:
            row.update({f: np.nan for f in ['max_envelope_exceedance_mt_yr', 'years_exceeding_envelope',
                                            'first_exceed_year', 'last_exceed_year', 'rmse_to_envelope_mt_yr']})
            row['envelope_consistent'] = False
        row['C_ratio_recon_to_scenario'] = row['C_recon_order_gt'] / row['C_scenario_gt']
        row['delta_tp_recon_vs_original'] = row['tp_recon_order'] - row['tp_original']
        row['delta_peak_year_recon_vs_screened'] = row['peak_year_recon_order'] - row['Y_peak_order']
        row['envelope_exceedance_threshold_Gt_yr'] = EXCEED_EPS_GT
        rows.append(row)
        original_stock = anchored_curve_values(row['model'], row['C_scenario_gt'], row['r_original'], row['S0_gt'], YEARS)[0]
        series = [('original', path['raw_rate_mt_yr'].to_numpy(float), original_stock, 'analytic_model_stock'),
                  ('screened_envelope', path['screened_rate_mt_yr'].to_numpy(float),
                   row['S0_gt'] + path['screened_cumulative_gt'].to_numpy(float),
                   'initial_stock_plus_allocation_level_integral')]
        if forward is not None:
            series.append(('reconstructed', forward['rate'] * 1000.0, forward['cum'], 'analytic_model_stock'))
        for kind, rate, stock, basis in series:
            paths.append(pd.DataFrame({**{k: row[k] for k in KEY}, 'year': YEARS, 'curve_kind': kind,
                                       'rate_mt_yr': rate, 'cum_gt': stock, 'cum_gt_includes_initial_stock': True,
                                       'curve_cumulative_basis': basis}))

    rec = pd.DataFrame(rows)
    solved = ~rec['landmark_infeasible']
    growth_ok = pd.Series(pd.NA, index=rec.index, dtype='boolean')
    growth_ok[solved] = rec.loc[solved, 'g_recon_order'] <= rec.loc[solved, 'g_cap']
    rec['growth_cap_criterion'] = 'reconstructed_cagr_le_scenario_cap'
    rec['landmark_solved'] = solved
    rec['growth_cap_pass'] = growth_ok
    rec['violates_g_cap'] = ~growth_ok

    rescreens, allocations, rescreen_paths = [], [], []
    for row in rec.loc[solved].to_dict('records'):
        rate = forward_r(row['model'], row['C_recon_order_gt'], row['S0_gt'], row['r_recon_order'])['rate'] * 1000.0
        checked, assigned, screened = screen(basins, row, row['allocation_order'], rate, {'pathway': 'reconstructed'})
        rescreens.append({**checked, 'rescreen_status': 'ok', 'rescreen_pass': checked['allocation_pass']})
        allocations.append(assigned.assign(pathway='reconstructed'))
        rescreen_paths.append(screened.assign(pathway='reconstructed'))
    rescreen = pd.DataFrame(rescreens)
    allocation_ok = rec[KEY].merge(rescreen[KEY + ['allocation_pass']], on=KEY, how='left')['allocation_pass']
    combined = rec['growth_cap_pass'] & allocation_ok.astype('boolean').array
    combined[~solved] = False
    rec['allocation_and_growth_cap_pass'] = combined
    rec['allocation_and_growth_cap_criterion'] = 'landmark_solved_and_allocation_pass_and_growth_cap_pass'
    names = {c: c if c.startswith('rescreen_') else f'rescreen_{c}' for c in rescreen.columns if c not in KEY}
    rec = rec.merge(rescreen[KEY + list(names)].rename(columns=names), on=KEY, how='left', validate='one_to_one')
    rec.loc[~rec['landmark_solved'], 'rescreen_status'] = 'not_run_unsolved'
    rec['rescreen_pass'] = rec['rescreen_pass'].astype('boolean')

    tables = {'reconstruction_summary': rec,
              'reconstruction_paths': pd.concat(paths, ignore_index=True),
              'reconstructed_screened_paths': pd.concat(rescreen_paths, ignore_index=True),
              'reconstructed_basin_allocation': pd.concat(allocations, ignore_index=True)}
    for name, table in tables.items():
        table.to_csv(out / f'{name}.csv', index=False)
    return rec

## Geological lifetime caps

In [ ]:
def lifetime_caps(boundary):
    out = OUTPUT / boundary
    summary = pd.read_csv(out / 'screening_summary.csv', float_precision='round_trip')
    rec = pd.read_csv(out / 'reconstruction_summary.csv', float_precision='round_trip')
    screened, reconstructed = dict(list(summary.groupby(KEYS))), dict(list(rec.groupby(KEYS)))
    rows = []
    for source in central.itertuples():
        key = tuple(getattr(source, k) for k in KEYS)
        passed = bool(screened[key]['allocation_pass'].all())
        rc = reconstructed.get(key)
        values = np.array([]) if rc is None else rc['C_recon_order_gt'].to_numpy(float)
        unresolved = 0 if rc is None else int((~rc['landmark_solved']).sum())
        available = passed or (unresolved == 0 and len(values) > 0 and np.isfinite(values).all())
        finite = values[np.isfinite(values)]
        if passed:
            cap, cap_source, reason = float(source.C_scenario_Gt), 'scenario_resource_both_orders_pass', ''
        elif available:
            cap, cap_source, reason = float(values.min()), 'minimum_failed_order_reconstruction', ''
        else:
            cap, cap_source = np.nan, 'unavailable_unresolved_reconstruction'
            reason = '; '.join(f'{r.allocation_order}: {r.landmark_reason}' for r in
                               rc.loc[~rc['landmark_solved']].sort_values('allocation_order').itertuples())
        growth_complete = rc is None or bool(rc['violates_g_cap'].notna().all())
        growth_count = 0 if rc is None else int(rc['violates_g_cap'].eq(True).sum())
        rows.append({**dict(zip(KEYS, key)), 'boundary': boundary, 'both_orders_pass': passed,
                     'C_scenario_gt': float(source.C_scenario_Gt), 'C_cap_gt': cap, 'cap_available': available,
                     'cap_source': cap_source, 'unavailable_reason': reason,
                     'C_recon_min_gt': float(finite.min()) if len(finite) else np.nan,
                     'C_recon_mean_gt': float(finite.mean()) if len(finite) else np.nan,
                     'reconstruction_order_count': len(values), 'unresolved_order_count': unresolved,
                     'has_growth_cap_violation': bool(growth_count) if growth_complete or growth_count else pd.NA,
                     'growth_cap_violating_order_count': growth_count,
                     'growth_cap_audit_complete': growth_complete,
                     'cap_interpretation': 'geological_lifetime_C_filter'})
    caps = pd.DataFrame(rows)
    caps.to_csv(out / 'geological_lifetime_caps.csv', index=False)
    return caps

## Run both boundaries

In [ ]:
for boundary in BOUNDARIES:
    start = time.time()
    basins = load_basins(boundary)
    run_screening(basins)
    rec = reconstruct(basins)
    caps = lifetime_caps(boundary)
    summary = pd.read_csv(OUTPUT / boundary / 'screening_summary.csv')
    print(f'{boundary}: {len(summary)} allocations, {int((~summary.allocation_pass).sum())} fail; '
          f'{len(rec)} reconstructions, {int(rec.landmark_solved.sum())} solved, '
          f'{int(rec.rescreen_pass.eq(True).sum())} pass rescreening; '
          f'{int(caps.cap_available.sum())}/120 caps ({time.time() - start:.0f} s)')
    unresolved = caps.loc[~caps.cap_available, KEYS + ['unavailable_reason']]
    if len(unresolved):
        print('Unresolved reconstructions (cap kept as NaN):')
        display(unresolved)

## Scope

The CO2BLOCK files are precomputed inputs. The well configurations cover only the basin and period combinations reached by the published pathways, so changed inputs may require regenerating them with CO2BLOCK.